In [5]:
import numpy as np

import matplotlib
matplotlib.use('TkAgg') 
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

from scipy import signal
from scipy.signal import ShortTimeFFT
from scipy.signal.windows import gaussian

import time
import os
import sys
import socket

In [6]:
def stft(fs, data, window, hop):
    
    SFT = ShortTimeFFT(window, hop=hop, fs=fs, mfft=200, scale_to='magnitude')
    Sx = SFT.stft(data)  # perform the STFT
    return Sx, SFT

In [ ]:
import sys
import time
import socket
import numpy as np
import scipy.signal as signal
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

def open_spectrograph():
    # ==========================================
    # 1. PARAMETRY SYSTEMOWE I FIZYCZNE
    # ==========================================
    FS = 44100            
    CHUNK_SIZE = 2048     
    NPERSEG = 512         
    # Decymacja rozmiaru bufora w celu redukcji obciążenia IPC
    HISTORY_LEN = 100     

    # Zastosowanie INADDR_ANY rozwiązuje EADDRNOTAVAIL (Errno 99)
    BIND_IP = "0.0.0.0"
    PORT = 5005

    TIME_PER_FRAME = CHUNK_SIZE / FS
    TOTAL_HISTORY_TIME = HISTORY_LEN * TIME_PER_FRAME

    freq_bins = NPERSEG // 2 + 1
    
    # Inicjalizacja bufora kołowego (z typowaniem float32 dla zmniejszenia RAM)
    waterfall = np.full((freq_bins, HISTORY_LEN), -100.0, dtype=np.float32)
    write_ptr = 0

    y_freq = np.linspace(0, FS/2, freq_bins)
    
    # ==========================================
    # 2. INICJALIZACJA INTERFEJSU (WebGL + IPyWidgets)
    # ==========================================
    # Użycie Heatmapgl wymusza sprzętową akcelerację renderowania w przeglądarce klienta
    heatmap = go.Heatmap(
        z=waterfall,
        y=y_freq,
        x0=-TOTAL_HISTORY_TIME,
        dx=TIME_PER_FRAME,
        colorscale='inferno',
        zmin=-100,
        zmax=0,
        colorbar=dict(title='Amplituda [dB]')
    )

    fig = go.FigureWidget(data=[heatmap])
    
    fig.update_layout(
        title="Real-Time STFT (WebGL + Ring Buffer)",
        yaxis_title="Częstotliwość [Hz]",
        xaxis_title="Czas bezwzględny [s]",
        width=900,
        height=600,
        margin=dict(l=60, r=20, t=50, b=50)
    )

    range_slider = widgets.FloatRangeSlider(
        value=[-100.0, 0.0],
        min=-150.0,
        max=100.0,
        step=1.0,
        description='Zakres dB:',
        continuous_update=True,
        layout=widgets.Layout(width='80%')
    )

    def on_slider_change(change):
        fig.data[0].zmin = change['new'][0]
        fig.data[0].zmax = change['new'][1]

    range_slider.observe(on_slider_change, names='value')

    ui_container = widgets.VBox([fig, range_slider])
    display(ui_container)

    # ==========================================
    # 3. INICJALIZACJA KOMUNIKACJI IPC (UDP)
    # ==========================================
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    try:
        sock.bind((BIND_IP, PORT))
        sock.setblocking(False)
        print(f"[SYSTEM ODBIORCZY] Gniazdo systemowe aktywne. Nasłuch na {BIND_IP}:{PORT}")
    except OSError as e:
        print(f"[BŁĄD KRYTYCZNY] Zakończono kod błędu jądra: {e}")
        sys.exit(1)

    # ==========================================
    # 4. PĘTLA WYKONAWCZA CZASU RZECZYWISTEGO
    # ==========================================
    total_frames = 0
    last_draw_time = time.time()
    
    # Redukcja do 15 FPS optymalizuje wykorzystanie protokołu WebSocket
    DRAW_INTERVAL = 1.0 / 15.0  

    try:
        while True:
            processed_any = False
            
            # Drenaż sprzętowego bufora warstwy łącza danych (L2) i transportowej (L4)
            while True:
                try:
                    data = sock.recv(CHUNK_SIZE * 4) 
                    chunk = np.frombuffer(data, dtype=np.float32)
                    
                    _, _, Zxx = signal.stft(chunk, fs=FS, nperseg=NPERSEG)
                    slice_idx = Zxx.shape[1] // 2
                    magnitude_db = 20 * np.log10(np.abs(Zxx[:, slice_idx]) + 1e-9) 
                    
                    # Zapis do bufora kołowego przy użyciu arytmetyki wskaźników
                    waterfall[:, write_ptr] = magnitude_db
                    write_ptr = (write_ptr + 1) % HISTORY_LEN
                    
                    total_frames += 1
                    processed_any = True
                    
                except BlockingIOError:
                    break 
                    
            # --- AKTUALIZACJA WIZUALIZACJI ---
            if processed_any:
                current_time = time.time()
                
                if (current_time - last_draw_time) >= DRAW_INTERVAL:
                    sim_time = total_frames * TIME_PER_FRAME
                    start_time = sim_time - TOTAL_HISTORY_TIME
                    
                    # Liniowa rekonstrukcja bloków pamięci do renderowania macierzy (kopiowanie wektora)
                    ordered_waterfall = np.concatenate((waterfall[:, write_ptr:], waterfall[:, :write_ptr]), axis=1)
                    
                    # Batch update serializuje zmiany i przesyła jeden pakiet JSON
                    with fig.batch_update():
                        fig.data[0].z = ordered_waterfall
                        fig.data[0].x0 = start_time
                        
                    last_draw_time = current_time
            
            # Wymuszenie oddania czasu procesora dla wątku obsługi zdarzeń Jupyter
            time.sleep(0.005)

    except KeyboardInterrupt:
        print("\n[SYSTEM] Przechwycono sygnał SIGINT. Zwalnianie deskryptorów...")
        
    finally:
        # Zabezpieczenie zwolnienia gniazda UDP nawet w przypadku wyjątku
        sock.close()
        print("[SYSTEM] Zwolniono gniazdo UDP. Moduł zatrzymany.")

# Wywołanie funkcji
# open_spectrograph()

In [ ]:
from pynq import Overlay, allocate

def open_spectrograph_fpga(window_name="Hann"):
    # ==========================================
    # 1. PARAMETRY SYSTEMOWE I FIZYCZNE
    # ==========================================
    FS = 44100            
    CHUNK_SIZE = 2048     
    NPERSEG = 512         
    HISTORY_LEN = 100     

    BIND_IP = "0.0.0.0"
    PORT = 5005

    TIME_PER_FRAME = CHUNK_SIZE / FS
    TOTAL_HISTORY_TIME = HISTORY_LEN * TIME_PER_FRAME

    freq_bins = NPERSEG // 2 + 1
    
    # Inicjalizacja bufora kołowego
    waterfall = np.full((freq_bins, HISTORY_LEN), -100.0, dtype=np.float32)
    write_ptr = 0

    y_freq = np.linspace(0, FS/2, freq_bins)
    
    # ==========================================
    # 2. INICJALIZACJA INTERFEJSU (WebGL + IPyWidgets)
    # ==========================================
    heatmap = go.Heatmap(
        z=waterfall,
        y=y_freq,
        x0=-TOTAL_HISTORY_TIME,
        dx=TIME_PER_FRAME,
        colorscale='inferno',
        zmin=-100,
        zmax=0,
        colorbar=dict(title='Amplituda [dB]')
    )

    fig = go.FigureWidget(data=[heatmap])
    
    fig.update_layout(
        title="Real-Time STFT (FPGA Acceleration)",
        yaxis_title="Częstotliwość [Hz]",
        xaxis_title="Czas bezwzględny [s]",
        width=900,
        height=600,
        margin=dict(l=60, r=20, t=50, b=50)
    )

    range_slider = widgets.FloatRangeSlider(
        value=[-100.0, 0.0],
        min=-150.0,
        max=100.0,
        step=1.0,
        description='Zakres dB:',
        continuous_update=True,
        layout=widgets.Layout(width='80%')
    )

    def on_slider_change(change):
        fig.data[0].zmin = change['new'][0]
        fig.data[0].zmax = change['new'][1]

    range_slider.observe(on_slider_change, names='value')

    ui_container = widgets.VBox([fig, range_slider])
    display(ui_container)

    # ==========================================
    # 3. INICJALIZACJA SPRZĘTU (PYNQ)
    # ==========================================
    print("[FPGA] Inicjalizacja Overlay...")
    ol = Overlay("design_main.bit")
    dma = ol.axi_dma_0
    gpio = ol.axi_gpio_0
    
    gpio.channel1.setdirection("out")
    
    # Reset sprzętowy IP
    gpio.channel1.write(0)
    time.sleep(0.01)
    
    # Wybór okna na podstawie window_name
    window_map = {
        "Hann": 1,
        "Blackman": 3,
        "Rectangular": 5
    }
    gpio_val = window_map.get(window_name, 1)
    print(f"[FPGA] Konfiguracja okna: {window_name} (GPIO wartość: {gpio_val})")
    gpio.channel1.write(gpio_val)
    
    # Alokacja buforów DMA (512 próbek wejściowych int16, 512 próbek wyjściowych int32)
    in_buffer = allocate(shape=(512,), dtype=np.int16)
    out_buffer = allocate(shape=(512,), dtype=np.int32)
    
    # Tablica indeksów do bit-reversalu
    bit_reversed_indices = []
    for i in range(512):
        rev = int('{:09b}'.format(i)[::-1], 2)
        bit_reversed_indices.append(rev)
        
    # ==========================================
    # 4. INICJALIZACJA KOMUNIKACJI IPC (UDP)
    # ==========================================
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    try:
        sock.bind((BIND_IP, PORT))
        sock.setblocking(False)
        print(f"[SYSTEM ODBIORCZY] Gniazdo systemowe aktywne. Nasłuch na {BIND_IP}:{PORT}")
    except OSError as e:
        print(f"[BŁĄD KRYTYCZNY] Zakończono kod błędu jądra: {e}")
        in_buffer.close()
        out_buffer.close()
        sys.exit(1)

    # ==========================================
    # 5. PĘTLA WYKONAWCZA CZASU RZECZYWISTEGO
    # ==========================================
    total_frames = 0
    last_draw_time = time.time()
    DRAW_INTERVAL = 1.0 / 15.0  

    try:
        while True:
            processed_any = False
            
            while True:
                try:
                    data = sock.recv(CHUNK_SIZE * 4) 
                    chunk = np.frombuffer(data, dtype=np.float32)
                    
                    # Wycinamy środkowy segment 512 próbek z 2048 próbek chunk
                    sub_chunk = chunk[768:1280]
                    
                    # Konwersja float32 do int16 i zapis do bufora PYNQ
                    in_buffer[:] = np.clip(sub_chunk * 32767.0, -32768, 32767).astype(np.int16)
                    
                    # Przeprowadzenie transferu DMA
                    dma.sendchannel.transfer(in_buffer)
                    dma.recvchannel.transfer(out_buffer)
                    dma.sendchannel.wait()
                    dma.recvchannel.wait()
                    
                    # Odczyt i rekonstrukcja danych zespolonych (LSB: Real, MSB: Imag)
                    fft_out = out_buffer.view(np.int16).reshape(-1, 2)
                    complex_data = fft_out[:, 0] + 1j * fft_out[:, 1]
                    
                    # Bit-reversal widma
                    complex_data = complex_data[bit_reversed_indices]
                    
                    # Skalowanie amplitudy
                    magnitude_db = 20 * np.log10(np.abs(complex_data[:257]) / (32767.0 * 128.0) + 1e-9)
                    
                    # Zapis do bufora kołowego
                    waterfall[:, write_ptr] = magnitude_db
                    write_ptr = (write_ptr + 1) % HISTORY_LEN
                    
                    total_frames += 1
                    processed_any = True
                    
                except BlockingIOError:
                    break 
                    
            # --- AKTUALIZACJA WIZUALIZACJI ---
            if processed_any:
                current_time = time.time()
                
                if (current_time - last_draw_time) >= DRAW_INTERVAL:
                    sim_time = total_frames * TIME_PER_FRAME
                    start_time = sim_time - TOTAL_HISTORY_TIME
                    
                    ordered_waterfall = np.concatenate((waterfall[:, write_ptr:], waterfall[:, :write_ptr]), axis=1)
                    
                    with fig.batch_update():
                        fig.data[0].z = ordered_waterfall
                        fig.data[0].x0 = start_time
                        
                    last_draw_time = current_time
            
            time.sleep(0.005)

    except KeyboardInterrupt:
        print("\n[SYSTEM] Przechwycono sygnał SIGINT. Zwalnianie deskryptorów...")
        
    finally:
        sock.close()
        in_buffer.close()
        out_buffer.close()
        print("[SYSTEM] Zwolniono gniazdo UDP oraz bufory PYNQ. Moduł zatrzymany.")


In [7]:
# when quitting matpliotlib close sockets and whatnot

In [8]:
open_spectrograph()

[SYSTEM ODBIORCZY] Gniazdo systemowe aktywne. Nasłuch na 10.0.0.1:5005

[SYSTEM] Otrzymano sygnał przerwania. Rozpoczynanie zwalniania zasobów...
[SYSTEM] Zakończono strumieniowanie. Deskryptor pliku został usunięty.


In [ ]:
# Uruchomienie funkcji ze sprzętowym STFT na FPGA
# Domyślnie z oknem Hann (wartość 1)
open_spectrograph_fpga(window_name="Hann")


In [10]:
%%bash --bg
# Zapis PID powłoki
echo $$ > /tmp/mic_tx.pid

# Aktywacja środowiska i przejście do katalogu roboczego
source ../bin/activate
cd src
python -u net_tx_mic.py

In [11]:
# Odczyt PID, wysłanie sygnału SIGTERM i usunięcie wskaźników
!kill -15 $(cat /tmp/mic_tx.pid) && rm /tmp/mic_tx.pid && echo "[SYSTEM] Proces zakończony. Logi pozostały w /tmp/mic_tx.log"


[SYSTEM] Proces zakończony. Logi pozostały w /tmp/mic_tx.log
